<a href="https://colab.research.google.com/github/asenin592/Programacion-de-Sistemas-de-Base-1/blob/main/Unidad%203/Colab/Colab2/GONZ%C3%81LEZ%20FRANCISCO%20PSSB%20I%20analizador_lexico_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GUÍA PRÁCTICA Y EVALUACIÓN ASÍNCRONA: ANALIZADORES LÉXICOS EN PYTHON Y JAVA
**Tema: Generación Automática de Analizadores Léxicos**  
**Asignatura:** Programación de Sistemas de Bas 1  

---  
## 🎯 Objetivos de Aprendizaje
1. **Comprender** la generación automática de analizadores léxicos a partir de especificaciones formales y expresiones regulares.
2. **Implementar y extender** escáneres léxicos en dos entornos tecnológicos representativos: **PLY (`ply.lex`)** en Python y **JFlex** en Java.
3. **Analizar** los mecanismos de disambiguación léxica (palabras reservadas vs. identificadores, coincidencia más larga) y el manejo de atributos/posiciones.
4. **Evaluar** estructuralmente las diferencias entre enfoques interpretados/dinámicos (PLY) y compilados/estáticos (JFlex).

---  
## 🐍 PARTE 1: Analizador Léxico en Python utilizando PLY (`ply.lex`)

### 1.1 Código Base e Instalación

In [ ]:
# Instalación de la librería PLY en el entorno de Google Colab
!pip install ply > /dev/null

In [ ]:
%%writefile lexer_ply.py

import ply.lex as lex

class PythonLexerPLY:
    """Especificación e implementación de un Analizador Léxico en Python usando PLY."""

    # 1. LISTA OBLIGATORIA DE TOKENS
    tokens = (
        'KEYWORD_IF',
        'KEYWORD_ELSE',
        'KEYWORD_WHILE',
        'KEYWORD_INT',
        'KEYWORD_FLOAT',
        'IDENTIFIER',
        'INTEGER_LITERAL',
        'FLOAT_LITERAL',
        'ASSIGN',
        'PLUS',
        'MINUS',
        'TIMES',
        'DIVIDE',
        'SEMICOLON',
        'LPAREN',
        'RPAREN',
        'LBRACE',
        'RBRACE',
    )

    # 2. DICCIONARIO ASOCIATIVO DE PALABRAS RESERVADAS
    reserved = {
        'if': 'KEYWORD_IF',
        'else': 'KEYWORD_ELSE',
        'while': 'KEYWORD_WHILE',
        'int': 'KEYWORD_INT',
        'float': 'KEYWORD_FLOAT'
    }

    # 3. REGLAS SIMPLES DE CADENA (t_TOKEN)
    t_ASSIGN    = r'='
    t_PLUS      = r'\+'
    t_MINUS     = r'-'
    t_TIMES     = r'\*'
    t_DIVIDE    = r'/'
    t_SEMICOLON = r';'
    t_LPAREN    = r'\('
    t_RPAREN    = r'\)'
    t_LBRACE    = r'\{'
    t_RBRACE    = r'\}'

    # Caracteres ignorados (espacios y tabuladores)
    t_ignore = ' \t'

    # 4. REGLAS BASADAS EN FUNCIONES (t_TOKEN(t))
    def t_FLOAT_LITERAL(self, t):
        r'\d+\.\d+([eE][+-]?\d+)?'
        t.value = float(t.value)
        return t

    def t_INTEGER_LITERAL(self, t):
        r'\d+'
        t.value = int(t.value)
        return t

    def t_IDENTIFIER(self, t):
        r'[a-zA-Z_][a-zA-Z0-9_]*'
        # Disambiguación entre palabras reservadas e identificadores de usuario
        t.type = self.reserved.get(t.value, 'IDENTIFIER')
        return t

    def t_newline(self, t):
        r'\n+'
        t.lexer.lineno += len(t.value)

    def t_error(self, t):
        print(f"⚠️  [Error Léxico] Línea {t.lexer.lineno}, Posición {t.lexpos}: Carácter ilegal '{t.value[0]}'")
        t.lexer.skip(1)

    def build(self, **kwargs):
        """Construye el objeto lexer dinámico mediante PLY."""
        self.lexer = lex.lex(object=self, **kwargs)

    def test(self, data: str):
        """Prueba el escáner sobre una cadena fuente imprimiendo los tokens capturados."""
        self.lexer.input(data)
        print(f"{'Línea':<8} | {'Tipo de Token':<18} | {'Lexema / Valor':<20} | {'Posición Abs.':<12}")
        print("-" * 65)
        for tok in self.lexer:
            print(f"{tok.lineno:<8} | {tok.type:<18} | {str(tok.value):<20} | {tok.lexpos:<12}")

# Práctica Base
sample_code = """
int contador = 10;
float promedio = 95.5;
while (contador) {
    promedio = promedio + 1.2e2;
    if (promedio) {
        @
    }
}
"""

#print("=== INICIANDO PRUEBA DEL ANALIZADOR LÉXICO EN PLY (Python) ===")
#py_lexer = PythonLexerPLY()
#py_lexer.build()
#py_lexer.test(sample_code)

Overwriting lexer_ply.py


In [ ]:
from lexer_ply import PythonLexerPLY

print("=== INICIANDO PRUEBA DEL ANALIZADOR LÉXICO EN PLY (Python) ===")
py_lexer = PythonLexerPLY()
py_lexer.build()
py_lexer.test(sample_code)

=== INICIANDO PRUEBA DEL ANALIZADOR LÉXICO EN PLY (Python) ===
Línea    | Tipo de Token      | Lexema / Valor       | Posición Abs.
-----------------------------------------------------------------
1        | KEYWORD_INT        | int                  | 0           
1        | IDENTIFIER         | contador             | 4           
1        | ASSIGN             | =                    | 13          
1        | INTEGER_LITERAL    | 10                   | 15          
1        | SEMICOLON          | ;                    | 17          
2        | KEYWORD_FLOAT      | float                | 19          
2        | IDENTIFIER         | promedio             | 25          
2        | ASSIGN             | =                    | 34          
2        | FLOAT_LITERAL      | 95.5                 | 36          
2        | SEMICOLON          | ;                    | 40          
3        | KEYWORD_WHILE      | while                | 42          
3        | LPAREN             | (                    |

### 🛠️ Desafío 1 (PLY - Python)
**Instrucciones:** Escribe a continuación la extensión de la clase `PythonLexerPLY` (o redefine la clase ampliada `PythonLexerPLYExtended`) para incluir:
1. Soporte para **cadenas literales** entrecomilladas (`"..."`) asignando el token `'STRING_LITERAL'`.
2. Soporte para **comentarios de una línea** (`//...`), configurando el lexer para que sean **ignorados en silencio** sin emitir ningún token.

In [ ]:
# [ESPACIO PARA EL ALUMNO] - Escribe tu código ampliado de PLY aquí
# Pista: Agrega 'STRING_LITERAL' a tokens y define t_STRING_LITERAL y t_ignore_COMMENT
%%writefile lexer_ply_ext.py

from lexer_ply import PythonLexerPLY

class PythonLexerPLYExtended(PythonLexerPLY):
    """Extensión del lexer: cadenas literales y comentarios de una línea."""

    # 1. Se agrega el nuevo token a la lista heredada
    tokens = PythonLexerPLY.tokens + ('STRING_LITERAL',)

    # 2. Comentarios de una línea (//...): se descartan en silencio.
    #    El prefijo t_ignore_ le indica a PLY que reconozca el patrón
    #    pero que NO emita ningún token.
    t_ignore_COMMENT = r'//.*'

    # 3. Cadenas literales entre comillas dobles ("...").
    #    Admite secuencias de escape como \" o \n dentro de la cadena.
    def t_STRING_LITERAL(self, t):
        r'"([^"\\\n]|\\.)*"'
        t.value = t.value[1:-1]   # se guarda el contenido sin las comillas
        return t


# Código de prueba para el desafío
sample_code_ext = """
// Programa de prueba con cadenas y comentarios
int contador = 10;          // contador inicial
float promedio = 95.5 / 2;
while (contador) {
    mensaje = "Hola mundo";
    ruta = "esto // no es comentario";
    escape = "dijo \\"hola\\"";
}
"""

Writing lexer_ply_ext.py


In [ ]:
import importlib
import lexer_ply_ext
importlib.reload(lexer_ply_ext)

from lexer_ply_ext import PythonLexerPLYExtended, sample_code_ext

print("=== PRUEBA DEL ANALIZADOR LÉXICO EXTENDIDO ===")
ext_lexer = PythonLexerPLYExtended()
ext_lexer.build()
ext_lexer.test(sample_code_ext)

=== PRUEBA DEL ANALIZADOR LÉXICO EXTENDIDO ===
Línea    | Tipo de Token      | Lexema / Valor       | Posición Abs.
-----------------------------------------------------------------
3        | KEYWORD_INT        | int                  | 49          
3        | IDENTIFIER         | contador             | 53          
3        | ASSIGN             | =                    | 62          
3        | INTEGER_LITERAL    | 10                   | 64          
3        | SEMICOLON          | ;                    | 66          
4        | KEYWORD_FLOAT      | float                | 97          
4        | IDENTIFIER         | promedio             | 103         
4        | ASSIGN             | =                    | 112         
4        | FLOAT_LITERAL      | 95.5                 | 114         
4        | DIVIDE             | /                    | 119         
4        | INTEGER_LITERAL    | 2                    | 121         
4        | SEMICOLON          | ;                    | 122         
5 

---  
## ☕ PARTE 2: Analizador Léxico en Java utilizando JFlex

### 2.1 Configuración e Instalación del Entorno

In [ ]:
import os

print("=== 1. INSTALANDO JDK EN GOOGLE COLAB ===")
ret = os.system("apt-get update -qq && apt-get install -y openjdk-11-jdk-headless > /dev/null")
print("✅ Entorno Java 11 configurado correctamente.\n" if ret == 0 else "❌ Falló la instalación del JDK.\n")

=== 1. INSTALANDO JDK EN GOOGLE COLAB ===
✅ Entorno Java 11 configurado correctamente.



### 2.2 Creación de Archivos Base de Java y Especificación JFlex

In [ ]:
# --- PASO 1: DEFINICIÓN DE LA ENUMERACIÓN DE TOKENS (TokenType.java) ---
tokentype_code = """
public enum TokenType {
    // Palabras Reservadas
    KEYWORD_IF, KEYWORD_ELSE, KEYWORD_WHILE, KEYWORD_INT, KEYWORD_FLOAT,

    // Identificadores y Literales
    IDENTIFIER, INTEGER_LITERAL, FLOAT_LITERAL, STRING_LITERAL,

    // Operadores y Delimitadores
    ASSIGN, PLUS, MINUS, TIMES, DIVIDE, SEMICOLON,
    LPAREN, RPAREN, LBRACE, RBRACE,

    // Control de Errores
    ERROR
}
"""

with open("TokenType.java", "w") as f:
    f.write(tokentype_code)

# --- PASO 2: CLASE CONTENEDORA DE TOKENS CON RASTREO (Token.java) ---
token_code = """
public class Token {
    public TokenType type;
    public Object value;
    public int line;
    public int column;

    public Token(TokenType type, Object value, int line, int column) {
        this.type = type;
        this.value = value;
        this.line = line;
        this.column = column;
    }

    @Override
    public String toString() {
        return String.format("%-8d | %-8d | %-18s | %-20s", line, column, type, value);
    }
}
"""

with open("Token.java", "w") as f:
    f.write(token_code)

# --- PASO 3: ARCHIVO DE ESPECIFICACIÓN JFLEX (scanner.jflex) ---
jflex_spec_code = """
import java.io.*;

%%

/* SECCIÓN 1: OPCIONES Y DIRECTIVAS DE CONFIGURACIÓN */
%class Lexer
%public
%unicode
%line
%column
%type Token

%{
    // Métodos auxiliares para construir objetos Token con coordenadas exactas
    private Token symbol(TokenType type) {
        return new Token(type, yytext(), yyline + 1, yycolumn + 1);
    }

    private Token symbol(TokenType type, Object value) {
        return new Token(type, value, yyline + 1, yycolumn + 1);
    }
%}

/* SECCIÓN 2: MACROS Y DEFINICIONES REGULARES */
DIGITO      = [0-9]
LETRA       = [a-zA-Z_]
ID          = {LETRA}({LETRA}|{DIGITO})*
ENTERO      = {DIGITO}+
FLOTANTE    = {DIGITO}+\\.{DIGITO}+([eE][+-]?{DIGITO}+)?
WS          = [ \\t\\r\\n]+

%%

/* SECCIÓN 3: REGLAS LÉXICAS Y ACCIONES EN JAVA */

/* Ignorar espacios en blanco, tabuladores y saltos de línea */
{WS}            { /* Ignorar */ }

/* Palabras Reservadas (prioridad por ubicación superior a ID) */
"if"            { return symbol(TokenType.KEYWORD_IF); }
"else"          { return symbol(TokenType.KEYWORD_ELSE); }
"while"         { return symbol(TokenType.KEYWORD_WHILE); }
"int"           { return symbol(TokenType.KEYWORD_INT); }
"float"         { return symbol(TokenType.KEYWORD_FLOAT); }

/* Literales con conversión de tipo nativo */
{FLOTANTE}      { return symbol(TokenType.FLOAT_LITERAL, Double.parseDouble(yytext())); }
{ENTERO}        { return symbol(TokenType.INTEGER_LITERAL, Integer.parseInt(yytext())); }

/* Identificadores */
{ID}            { return symbol(TokenType.IDENTIFIER, yytext()); }

/* Operadores y Delimitadores */
"="             { return symbol(TokenType.ASSIGN); }
"+"             { return symbol(TokenType.PLUS); }
"-"             { return symbol(TokenType.MINUS); }
"*"             { return symbol(TokenType.TIMES); }
"/"             { return symbol(TokenType.DIVIDE); }
";"             { return symbol(TokenType.SEMICOLON); }
"("             { return symbol(TokenType.LPAREN); }
")"             { return symbol(TokenType.RPAREN); }
"{"             { return symbol(TokenType.LBRACE); }
"}"             { return symbol(TokenType.RBRACE); }

/* Captura de Errores Léxicos (Modo Pánico) */
.               {
                    System.err.printf("⚠️  [Error Léxico] Línea %d, Columna %d: Carácter ilegal '%s'\\n",
                                      yyline + 1, yycolumn + 1, yytext());
                    return symbol(TokenType.ERROR, yytext());
                }
"""

with open("scanner.jflex", "w") as f:
    f.write(jflex_spec_code)

# --- PASO 4: CLASE PRINCIPAL DE PRUEBA (Main.java) ---
main_code = """
import java.io.*;

public class Main {
    public static void main(String[] args) {
        if (args.length == 0) {
            System.out.println("Por favor ingrese la ruta del archivo de prueba.");
            return;
        }

        try {
            FileReader reader = new FileReader(args[0]);
            Lexer scanner = new Lexer(reader);

            System.out.println("=== ANALIZADOR LÉXICO GENERADO POR JFLEX (Java) ===");
            System.out.println(String.format("%-8s | %-8s | %-18s | %-20s", "Línea", "Columna", "Tipo de Token", "Lexema / Valor"));
            System.out.println("-".repeat(65));

            Token token;
            while ((token = scanner.yylex()) != null) {
                System.out.println(token);
            }
            System.out.println("\\n=== ESCANEO FINALIZADO EXITOSAMENTE ===");

        } catch (IOException e) {
            System.err.println("Error al leer el archivo: " + e.getMessage());
        }
    }
}
"""

with open("Main.java", "w") as f:
    f.write(main_code)

# --- PASO 5: ARCHIVO DE FUENTE DE PRUEBA (prueba.txt) ---
sample_code = """int contador = 10;
float promedio = 95.5;
while (contador) {
    promedio = promedio + 1.2e2;
    if (promedio) {
        @
    }
}
"""

with open("prueba.txt", "w") as f:
    f.write(sample_code)

print("✅ Archivos Java y especificacion scanner.jflex creados exitosamente.")

✅ Archivos Java y especificacion scanner.jflex creados exitosamente.


### 2.3 Compilación y Ejecución en Colab

In [ ]:
print("=== DESCARGANDO JFLEX (JAR con dependencias) ===")
!rm -rf jflex-1.9.1 jflex.tar.gz
!wget -q https://github.com/jflex-de/jflex/releases/download/v1.9.1/jflex-1.9.1.tar.gz -O jflex.tar.gz
!tar -xzf jflex.tar.gz

print("\n=== GENERANDO Y COMPILANDO ARCHIVOS JAVA CON JFLEX ===")
!java -Xlog:disable -jar jflex-1.9.1/lib/jflex-full-1.9.1.jar scanner.jflex
!javac -J-Xlog:disable TokenType.java Token.java Lexer.java Main.java

print("\n=== SALIDA DEL ESCÁNER PROCESANDO 'prueba.txt' ===")
!java -Xlog:disable Main prueba.txt

=== DESCARGANDO JFLEX (JAR con dependencias) ===

=== GENERANDO Y COMPILANDO ARCHIVOS JAVA CON JFLEX ===
Reading "scanner.jflex"
Constructing NFA : 80 states in NFA
Converting NFA to DFA : 
........................................
42 states before minimization, 38 states in minimized DFA
Writing code to "Lexer.java"

=== SALIDA DEL ESCÁNER PROCESANDO 'prueba.txt' ===
=== ANALIZADOR LÉXICO GENERADO POR JFLEX (Java) ===
Línea    | Columna  | Tipo de Token      | Lexema / Valor      
-----------------------------------------------------------------
1        | 1        | KEYWORD_INT        | int                 
1        | 5        | IDENTIFIER         | contador            
1        | 14       | ASSIGN             | =                   
1        | 16       | INTEGER_LITERAL    | 10                  
1        | 18       | SEMICOLON          | ;                   
2        | 1        | KEYWORD_FLOAT      | float               
2        | 7        | IDENTIFIER         | promedio            


### 🛠️ Desafío 2 (JFlex - Java)
**Instrucciones:** Modifica la especificación `scanner.jflex` arriba o vuelve a escribir el archivo desde una celda agregando:
1. La definición de la regla de **cadenas literales entrecomilladas**: `\"[^\"]*\" { return symbol(TokenType.STRING_LITERAL, yytext()); }`
2. La regla para **comentarios de una línea**: `"//".* { /* Ignorar */ }`
Recompila con `!jflex scanner.jflex && javac *.java` y ejecuta sobre un archivo de prueba ampliado.

In [ ]:
%%writefile scanner.jflex
import java.io.*;

%%

/* SECCIÓN 1: OPCIONES Y DIRECTIVAS DE CONFIGURACIÓN */
%class Lexer
%public
%unicode
%line
%column
%type Token

%{
    // Métodos auxiliares para construir objetos Token con coordenadas exactas
    private Token symbol(TokenType type) {
        return new Token(type, yytext(), yyline + 1, yycolumn + 1);
    }

    private Token symbol(TokenType type, Object value) {
        return new Token(type, value, yyline + 1, yycolumn + 1);
    }
%}

/* SECCIÓN 2: MACROS Y DEFINICIONES REGULARES */
DIGITO      = [0-9]
LETRA       = [a-zA-Z_]
ID          = {LETRA}({LETRA}|{DIGITO})*
ENTERO      = {DIGITO}+
FLOTANTE    = {DIGITO}+\.{DIGITO}+([eE][+-]?{DIGITO}+)?
WS          = [ \t\r\n]+

%%

/* SECCIÓN 3: REGLAS LÉXICAS Y ACCIONES EN JAVA */

/* Ignorar espacios en blanco, tabuladores y saltos de línea */
{WS}            { /* Ignorar */ }

/* [DESAFÍO 2] Comentarios de una línea: se ignoran en silencio */
"//".*          { /* Ignorar */ }

/* [DESAFÍO 2] Cadenas literales entrecomilladas */
\"[^\"]*\"      { return symbol(TokenType.STRING_LITERAL, yytext()); }

/* Palabras Reservadas (prioridad por ubicación superior a ID) */
"if"            { return symbol(TokenType.KEYWORD_IF); }
"else"          { return symbol(TokenType.KEYWORD_ELSE); }
"while"         { return symbol(TokenType.KEYWORD_WHILE); }
"int"           { return symbol(TokenType.KEYWORD_INT); }
"float"         { return symbol(TokenType.KEYWORD_FLOAT); }

/* Literales con conversión de tipo nativo */
{FLOTANTE}      { return symbol(TokenType.FLOAT_LITERAL, Double.parseDouble(yytext())); }
{ENTERO}        { return symbol(TokenType.INTEGER_LITERAL, Integer.parseInt(yytext())); }

/* Identificadores */
{ID}            { return symbol(TokenType.IDENTIFIER, yytext()); }

/* Operadores y Delimitadores */
"="             { return symbol(TokenType.ASSIGN); }
"+"             { return symbol(TokenType.PLUS); }
"-"             { return symbol(TokenType.MINUS); }
"*"             { return symbol(TokenType.TIMES); }
"/"             { return symbol(TokenType.DIVIDE); }
";"             { return symbol(TokenType.SEMICOLON); }
"("             { return symbol(TokenType.LPAREN); }
")"             { return symbol(TokenType.RPAREN); }
"{"             { return symbol(TokenType.LBRACE); }
"}"             { return symbol(TokenType.RBRACE); }

/* Captura de Errores Léxicos (Modo Pánico) */
.               {
                    System.err.printf("⚠️  [Error Léxico] Línea %d, Columna %d: Carácter ilegal '%s'\n",
                                      yyline + 1, yycolumn + 1, yytext());
                    return symbol(TokenType.ERROR, yytext());
                }

Overwriting scanner.jflex


In [ ]:
%%writefile prueba_ampliada.txt
// Programa de prueba ampliado (Desafío 2)
int contador = 10;            // contador inicial
float promedio = 95.5 / 2;
while (contador) {
    mensaje = "Hola mundo";
    ruta = "esto // no es comentario";
    vacio = "";
    promedio = promedio + 1.2e2;   // se suma 120.0
    if (promedio) {
        @
    }
}

Writing prueba_ampliada.txt


In [ ]:
print("=== REGENERANDO EL LEXER CON JFLEX (Desafío 2) ===")
!java -Xlog:disable -jar jflex-1.9.1/lib/jflex-full-1.9.1.jar scanner.jflex && javac -J-Xlog:disable -encoding UTF-8 *.java

print("\n=== SALIDA DEL ESCÁNER PROCESANDO 'prueba_ampliada.txt' ===")
!java -Xlog:disable Main prueba_ampliada.txt

=== REGENERANDO EL LEXER CON JFLEX (Desafío 2) ===
Reading "scanner.jflex"
Constructing NFA : 95 states in NFA
Converting NFA to DFA : 
.............................................
47 states before minimization, 42 states in minimized DFA
Old file "Lexer.java" saved as "Lexer.java~"
Writing code to "Lexer.java"

=== SALIDA DEL ESCÁNER PROCESANDO 'prueba_ampliada.txt' ===
=== ANALIZADOR LÉXICO GENERADO POR JFLEX (Java) ===
Línea    | Columna  | Tipo de Token      | Lexema / Valor      
-----------------------------------------------------------------
2        | 1        | KEYWORD_INT        | int                 
2        | 5        | IDENTIFIER         | contador            
2        | 14       | ASSIGN             | =                   
2        | 16       | INTEGER_LITERAL    | 10                  
2        | 18       | SEMICOLON          | ;                   
3        | 1        | KEYWORD_FLOAT      | float               
3        | 7        | IDENTIFIER         | promedio        

---  
## 📝 BITÁCORA Y REPORTE DE LABORATORIO

**Nombre del Estudiante:** González Mercado Francisco Antonio  
**Matrícula / Código:** a223133003  
**Fecha:** ___11/11/2026__  

---
### Sección 1: Verificación de Coincidencia y Resolución de Ambigüedades
1. **Mecanismo de la regla más larga (*Longest Match*):**  
   Al analizar la subcadena `1.2e2`, explica paso a paso por qué el autómata generado no la separa como el entero `1`, el operador `.` y el identificador/entero `2e2`, sino como un único token `FLOAT_LITERAL`.  
   *Respuesta del alumno:*
   >

>El autómata finito determinista (AFD) que genera JFlex no se detiene en el primer estado de aceptación que encuentra: sigue leyendo caracteres mientras exista una transición válida y va recordando la **última posición en la que estuvo en un estado de aceptación**. Para la entrada `1.2e2;` el recorrido es:

| Carácter leído | Prefijo reconocido | ¿Estado de aceptación? | Última aceptación guardada |
|---|---|---|---|
| `1` | `1` | Sí (ENTERO) | ENTERO, longitud 1 |
| `.` | `1.` | No (prefijo de FLOTANTE) | ENTERO, longitud 1 |
| `2` | `1.2` | Sí (FLOTANTE) | FLOTANTE, longitud 3 |
| `e` | `1.2e` | No (espera el exponente) | FLOTANTE, longitud 3 |
| `2` | `1.2e2` | Sí (FLOTANTE) | FLOTANTE, longitud 5 |
| `;` | — | No hay transición | Se detiene |

>Al no haber transición con `;`, el autómata devuelve la última aceptación registrada: el lexema `1.2e2` (5 caracteres) con la regla `{FLOTANTE}`. Por eso emite un solo `FLOAT_LITERAL`, y `Double.parseDouble` lo convierte a `120.0`. La división `1` + `.` + `2e2` nunca se elige, porque el entero `1` mide 1 carácter y la coincidencia más larga es de 5.

>Los estados intermedios `1.` y `1.2e` no son de aceptación, pero el autómata no se detiene porque todavía hay transiciones posibles. Si la entrada fuera `1.;`, no podría completar el flotante: **retrocedería** a la última aceptación (`ENTERO 1`) y el `.` se analizaría después como carácter ilegal. Además, `.` no es un operador del lenguaje y `2e2` no puede ser un identificador, porque `{ID}` debe empezar con una letra o `_`.

>En PLY el mismo resultado se logra de otra forma: el módulo `re` de Python toma la **primera** alternativa que coincide, no la más larga, así que `t_FLOAT_LITERAL` debe definirse antes que `t_INTEGER_LITERAL`.
   

2. **Disambiguación de Palabras Reservadas vs. Identificadores:**  
   * En **JFlex**, ¿por qué la posición (física) de las reglas de `"if"`, `"while"`, etc., por encima de la regla `{ID}` garantiza que no sean clasificadas como variables?
   * En **PLY**, ¿por qué se utiliza el método `reserved.get(t.value, 'IDENTIFIER')` dentro de la función `t_IDENTIFIER` en lugar de crear una regla independiente `t_KEYWORD_IF = r'if'`?  
   *Respuesta del alumno:*
   >
*Respuesta del alumno:*

**En JFlex:**

- Con la entrada `if`, las reglas `"if"` y `{ID}` coinciden con la **misma longitud** (2 caracteres), así que el criterio de la coincidencia más larga no decide.
- JFlex resuelve el empate eligiendo **la regla que aparece primero en la especificación**. En el AFD, el estado de aceptación al que se llega después de leer `if` pertenece a las dos reglas, y JFlex le asigna la acción de la que está más arriba. Por eso las palabras reservadas se colocan antes de `{ID}`.
- Esto solo afecta a los empates. Con `iffy`, la regla `{ID}` reconoce 4 caracteres contra 2 de `"if"`, gana por longitud y se clasifica correctamente como `IDENTIFIER`.
- Lo comprobé poniendo `{ID}` arriba de `"if"`: JFlex muestra la advertencia `Rule can never be matched` y la entrada `if` sale como `IDENTIFIER: if`.

**En PLY:**

- PLY no aplica coincidencia más larga entre reglas. Arma una sola expresión regular maestra con alternativas: primero las reglas definidas como función (en orden de definición) y después las de cadena (de la regex más larga a la más corta). El módulo `re` se queda con la **primera** alternativa que coincide.
- Por eso una regla independiente falla de cualquiera de estas dos formas (lo verifiqué con la entrada `if iffy`):
    - **Si se define como cadena** (`t_KEYWORD_IF = r'if'`), la función `t_IDENTIFIER` se prueba antes y siempre gana. La palabra reservada nunca se reconoce: `[IDENTIFIER 'if', IDENTIFIER 'iffy']`.
    - **Si se define como función antes de `t_IDENTIFIER`**, reconoce `if` como prefijo de otras palabras y parte los identificadores: `[KEYWORD_IF 'if', KEYWORD_IF 'if', IDENTIFIER 'fy']`.
- Con `reserved.get(t.value, 'IDENTIFIER')`, la regla `t_IDENTIFIER` primero reconoce **la palabra completa** y después la busca en un diccionario. Si la encuentra, cambia el tipo a la palabra reservada; si no, se queda como `IDENTIFIER`.
- Ventajas de este enfoque:
    - Elimina los problemas de orden y de prefijos.
    - La búsqueda en el diccionario es O(1).
    - La expresión regular maestra es más pequeña.
    - Agregar una palabra reservada nueva solo requiere una entrada en el diccionario.
    - Es el enfoque que recomienda la documentación de PLY.

---
### Sección 2: Conversión y Tipado de Atributos Léxicos
1. **Conversión de Tipos:**  
   En la regla `t_FLOAT_LITERAL` de PLY se ejecuta `t.value = float(t.value)`, y en JFlex se pasa `Double.parseDouble(yytext())`. ¿Por qué es fundamental que esta conversión ocurra dentro del analizador léxico antes de entregar el token al analizador sintáctico/semántico?  
   *Respuesta del alumno:*
   >

- **Un token es un par `<tipo, atributo>`.** El tipo (`FLOAT_LITERAL`) le basta al analizador sintáctico para validar la gramática, pero las fases semántica y de generación de código necesitan el **valor numérico real**, no el texto. El analizador léxico es la única fase que trabaja directamente con los caracteres, así que es el lugar natural para interpretarlos.
- **Se convierte una sola vez.** Si el token llevara la cadena `"1.2e2"`, cada fase posterior (verificación de tipos, plegado de constantes, generación de código) tendría que volver a analizar el texto. Eso es repetir trabajo y mezclar responsabilidades.
- **Normaliza el valor.** Escrituras distintas del mismo número (`1.2e2`, `120.0`, `120.00`) se vuelven el mismo valor (`120.0`). Las fases posteriores comparan y operan valores, no texto.
- **Habilita el análisis semántico.** Como el atributo ya es de tipo `float`/`Double` o `int`/`Integer`, el analizador semántico puede inferir el tipo de la expresión, detectar incompatibilidades (por ejemplo, asignar `95.5` a una variable `int`) y evaluar constantes en tiempo de compilación (`promedio + 1.2e2` ya opera con `120.0`).
- **Detecta errores en el lugar correcto.** Si el literal es inválido o no cabe en el tipo, el error aparece en la fase léxica, cuando todavía se conocen la línea y la columna exactas del lexema. Por ejemplo, en JFlex `Integer.parseInt("99999999999")` lanza `NumberFormatException` porque excede el rango de `int`. En Python no pasa porque `int` tiene precisión arbitraria. Esto también muestra que en un compilador real esa excepción debería capturarse y reportarse como error léxico.

---
### Sección 3: Análisis Estructural y Manejo de Posiciones
1. **Comparación de Rastreos de Coordenadas:**  
   Explica la diferencia técnica entre cómo rastrea PLY la ubicación de los tokens (utilizando `t.lexer.lineno` y `t.lexpos`) y cómo lo realiza JFlex mediante las directivas `%line` y `%column` con las variables internas `yyline` y `yycolumn`.  
   *Respuesta del alumno:*
   >

**En PLY:**

- **`t.lexpos`** es la posición absoluta del token: un índice que empieza en 0 y cuenta desde el inicio de la cadena de entrada. PLY la calcula **automáticamente**.
- **`t.lexer.lineno`** es solo un contador que **PLY no actualiza por sí mismo**. El programador debe incrementarlo en una regla, como `t_newline` con `t.lexer.lineno += len(t.value)`. Si se omite esa regla, todos los tokens reportan la línea 1. Si un token abarca varias líneas (una cadena o un comentario multilínea), también hay que contar sus saltos manualmente.
- **PLY no lleva columna.** Hay que calcularla a partir de `lexpos`, buscando el último salto de línea anterior:
    - `inicio_linea = data.rfind('\n', 0, t.lexpos) + 1`
    - `columna = t.lexpos - inicio_linea + 1`
- Ejemplo: el carácter `@` se reportó en la posición absoluta `123`. Con esa fórmula se obtiene **línea 7, columna 9**.

**En JFlex:**

- **`%line`** y **`%column`** son directivas que le indican a JFlex que **genere el código de conteo** dentro del escáner. Ese código incrementa `yyline` y reinicia `yycolumn` a 0 cada vez que consume un fin de línea (`\n`, `\r`, `\r\n`, etc.).
- Las variables **`yyline`** y **`yycolumn`** existen siempre en la clase generada, pero sin las directivas se quedan en 0, porque no se genera el código que las actualiza. El conteo es opcional porque añade un pequeño costo por cada carácter procesado.
- Ambas empiezan en 0. Por eso en `symbol()` se usa `yyline + 1` y `yycolumn + 1`.
- Siempre indican el **inicio del token actual**. La columna se obtiene directamente, sin cálculos extra.
- El equivalente a `lexpos` en JFlex es la directiva **`%char`**, que activa `yychar` (desplazamiento absoluto).

**Diferencia clave:**

- PLY da la **posición absoluta** de forma automática, pero la **línea es responsabilidad del programador** y la **columna se deriva manualmente**.
- JFlex **genera el rastreo de línea y columna automáticamente** con solo activar las directivas.
- Los dos llegan a la misma ubicación para el error del `@`: **columna 9**. La línea difiere (7 en PLY y 6 en JFlex) solo porque la cadena de prueba de PLY empieza con un salto de línea.
---

### Sección 4: Extensión de Reglas (Evidencia del Desafío)
Pega las líneas de código o expresiones regulares que agregaste en PLY y JFlex para dar soporte a cadenas literales y comentarios de una línea:

**Código/Reglas agregadas en PLY (Python):**

```python
# Pegar aquí el fragmento de código

class PythonLexerPLYExtended(PythonLexerPLY):
    """Extensión del lexer: cadenas literales y comentarios de una línea."""

    # 1. Se agrega el nuevo token a la lista heredada
    tokens = PythonLexerPLY.tokens + ('STRING_LITERAL',)

    # 2. Comentarios de una línea (//...): se descartan en silencio.
    #    El prefijo t_ignore_ le indica a PLY que reconozca el patrón
    #    pero que NO emita ningún token.
    t_ignore_COMMENT = r'//.*'

    # 3. Cadenas literales entre comillas dobles ("...").
    #    Admite secuencias de escape como \" o \n dentro de la cadena.
    def t_STRING_LITERAL(self, t):
        r'"([^"\\\n]|\\.)*"'
        t.value = t.value[1:-1]   # se guarda el contenido sin las comillas
        return t

```

**Reglas agregadas en scanner.jflex (Java):**

```jflex
/* Pegar aquí las reglas de JFlex */

class PythonLexerPLYExtended(PythonLexerPLY):
    """Extensión del lexer: cadenas literales y comentarios de una línea."""

    # 1. Se agrega el nuevo token a la lista heredada
    tokens = PythonLexerPLY.tokens + ('STRING_LITERAL',)

    # 2. Comentarios de una línea (//...): se descartan en silencio.
    #    El prefijo t_ignore_ le indica a PLY que reconozca el patrón
    #    pero que NO emita ningún token.
    t_ignore_COMMENT = r'//.*'

    # 3. Cadenas literales entre comillas dobles ("...").
    #    Admite secuencias de escape como \" o \n dentro de la cadena.
    def t_STRING_LITERAL(self, t):
        r'"([^"\\\n]|\\.)*"'
        t.value = t.value[1:-1]   # se guarda el contenido sin las comillas
        return t

```

---
### Sección 5: Cuadro Comparativo Arquitectónico

| Criterio de Comparación | PLY (Python) | JFlex (Java) |
| :--- | :--- | :--- |
| **Naturaleza del Generador** | (Interpretado / Reflexivo / Compilado) | (Interpretado / Reflexivo / Compilado) |
| **Definición de Patrones** | Docstrings en funciones Python | Archivo de especificación `.jflex` |
| **Manejo de Palabras Reservadas** | | |
| **Rastreo de Línea y Columna** | | |
| **Recuperación de Errores** | | |

---  


| Criterio de Comparación | PLY (Python) | JFlex (Java) |
|---|---|---|
| **Naturaleza del Generador** | **Reflexivo**<br>Inspecciona en ejecución las reglas `t_`<br>y arma una regex maestra con `re`.<br>No genera código fuente. | **Compilado**<br>Genera `Lexer.java` con un AFD minimizado<br>(42 estados en la práctica),<br>que luego se compila con `javac`. |
| **Definición de Patrones** | Docstrings en funciones Python<br>(`def t_TOKEN`) y variables<br>de cadena (`t_TOKEN = r'...'`). | Archivo de especificación `.jflex`<br>con macros (`DIGITO`, `ID`)<br>y reglas con acciones en Java. |
| **Manejo de Palabras Reservadas** | Diccionario `reserved` consultado<br>dentro de `t_IDENTIFIER`.<br>`re` toma la primera alternativa,<br>no la más larga. | Reglas literales (`"if"`, `"while"`)<br>antes de `{ID}`: gana la más larga<br>y, si empatan, la de más arriba. |
| **Rastreo de Línea y Columna** | `t.lexpos` es automático.<br>`lineno` se actualiza a mano<br>en `t_newline`; la columna<br>se calcula desde `lexpos`. | Automático con `%line` y `%column`<br>(`yyline`, `yycolumn`, base 0 → `+1`).<br>`%char` da la posición absoluta. |
| **Recuperación de Errores** | `t_error` reporta el carácter<br>y `t.lexer.skip(1)` lo descarta<br>(modo pánico). No emite token. | Regla comodín `.` al final:<br>reporta y emite un token `ERROR`.<br>Sin ella, JFlex lanza una excepción. |

**Conclusión General del Experimento:**  
> *(Redacta una breve conclusión sobre las ventajas y desventajas de cada herramienta en el desarrollo de un compilador real)*

> Ambas herramientas generaron los mismos tokens y detectaron el mismo error léxico (`@` en la columna 9), lo que confirma que se basan en los mismos fundamentos: expresiones regulares y autómatas finitos.
>
> **PLY** es más rápido de usar: todo se escribe en Python, no requiere generar ni compilar código y se integra con `ply.yacc`, por lo que es ideal para prototipos y fines didácticos. Su desventaja es un menor rendimiento y que deja al programador tareas como contar líneas, calcular columnas y cuidar el orden de las reglas.
>
> **JFlex** genera un AFD minimizado más eficiente, resuelve ambigüedades de forma nativa y rastrea línea y columna automáticamente, por lo que es más adecuado para un compilador real. A cambio, su flujo de trabajo es más pesado (`.jflex` → `Lexer.java` → `javac`) y depende de Java.
>
